# Project · Gemini Cinema Advisor — structured output + OMDb + LLM-as-judge

LangChain + Gemini return a **Pydantic-validated** recommendation; OMDb adds posters and ratings; a *separate* judge call scores the result. Here we run it without Streamlit.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |
| `OMDB_API_KEY` | OMDb key (free: 1,000 requests/day, emailed to you) | https://www.omdbapi.com/apikey.aspx |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q langchain langchain-google-genai pydantic requests python-dotenv

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

In [ ]:
%cd /content
!mkdir -p movie_advisor
%cd movie_advisor

In [ ]:
%%writefile tracing.py
"""
A very lightweight execution tracer.

Every meaningful step (which function ran, what it decided, how long an API call
took, what error it hit) gets logged here. `app.py` calls `start_trace()` once at
the top of handling a user message, every other module calls `log()` /
`log_exception()` as it works, and `app.py` renders `get_trace_text()` in a debug
expander at the end - so you get a full, ordered trace of exactly what happened
for that turn, without needing to dig through a terminal.

Note: this uses a module-level buffer, so it assumes one request is being traced
at a time per process - fine for a single-user local Streamlit app like this one,
but not safe for real concurrent multi-user traffic.
"""

from __future__ import annotations

import logging
import time
import traceback

logger = logging.getLogger("cinema_advisor")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

_records: list[str] = []


def start_trace() -> None:
    """Call once at the start of handling a user turn to reset the trace buffer."""
    _records.clear()
    log("--- new turn started ---")


def log(message: str, level: str = "INFO") -> None:
    ts = time.strftime("%H:%M:%S")
    _records.append(f"[{ts}] [{level}] {message}")
    if level == "ERROR":
        logger.error(message)
    else:
        logger.debug(message)


def log_exception(context: str) -> None:
    """Log the current exception (call this from inside an except block)."""
    tb = traceback.format_exc()
    log(f"EXCEPTION in {context}:\n{tb}", level="ERROR")


class timed:
    """Context manager: `with timed("omdb.get_movie"): ...` logs start/end + duration."""

    def __init__(self, label: str):
        self.label = label

    def __enter__(self):
        self.start = time.perf_counter()
        log(f"→ starting: {self.label}")
        return self

    def __exit__(self, exc_type, exc, tb):
        elapsed_ms = (time.perf_counter() - self.start) * 1000
        if exc_type is not None:
            log(f"✗ failed: {self.label} after {elapsed_ms:.0f}ms - {exc_type.__name__}: {exc}", level="ERROR")
        else:
            log(f"✓ finished: {self.label} in {elapsed_ms:.0f}ms")
        return False  # never suppress the exception


def get_trace_text() -> str:
    return "\n".join(_records) if _records else "(no trace recorded for this turn)"

In [ ]:
%%writefile models.py
"""
Pydantic models used across the app.

- MovieRecommendation / RecommendationResponse: the structured output we ask
  Gemini (via LangChain) to return.
- OmdbMovie: a typed wrapper around whatever OMDb's API gives us back.
"""

from __future__ import annotations

from typing import List, Optional

from pydantic import BaseModel, Field, field_validator


class MovieRecommendation(BaseModel):
    """A single movie suggestion produced by the LLM."""

    title: str = Field(description="The exact movie title, as it would appear on IMDb/OMDb.")
    year: Optional[int] = Field(
        default=None, description="Release year of the movie, if known."
    )
    reason: str = Field(
        description="A short, specific explanation of why this movie fits the user's request."
    )
    genres: List[str] = Field(
        default_factory=list, description="A few relevant genres for this movie."
    )


class RecommendationResponse(BaseModel):
    """The full structured response the LLM must return for a recommendation turn."""

    reply: str = Field(
        description="A short, conversational reply to show the user above the movie list."
    )
    movies: List[MovieRecommendation] = Field(
        default_factory=list,
        description="0-5 movie recommendations. Empty if you are just asking a clarifying question.",
    )
    follow_up_question: Optional[str] = Field(
        default=None,
        description="An optional follow-up question to help narrow down future recommendations.",
    )

    @field_validator("movies")
    @classmethod
    def _cap_movies(cls, v: List[MovieRecommendation]) -> List[MovieRecommendation]:
        return v[:5]


class MovieEvaluation(BaseModel):
    """An independent judge's score for a single recommended movie."""

    title: str = Field(description="Must exactly match the title being evaluated.")
    relevance_score: int = Field(
        description="An integer from 1 to 10: how well this specific movie fits the "
        "user's stated request/preferences.",
    )
    justification: str = Field(
        description="One short sentence on why it earned this score - be specific and critical."
    )

    @field_validator("relevance_score")
    @classmethod
    def _clamp_relevance(cls, v: int) -> int:
        # Gemini's structured-output schema converter doesn't reliably enforce
        # numeric min/max constraints (ge=/le= on the Field), so we clamp here in
        # Python instead of relying on the API to reject out-of-range values -
        # that also avoids the whole call failing if it returns e.g. 0 or 11.
        return max(1, min(10, v))


class EvaluationResponse(BaseModel):
    """Overall judge verdict for one recommendation turn."""

    overall_score: int = Field(
        description="An integer from 1 to 10: how well the recommendation set as a "
        "whole satisfies the user's request.",
    )
    overall_feedback: str = Field(
        description="1-2 sentences of honest, critical feedback on the recommendation set."
    )
    movie_evaluations: List[MovieEvaluation] = Field(default_factory=list)

    @field_validator("overall_score")
    @classmethod
    def _clamp_overall(cls, v: int) -> int:
        return max(1, min(10, v))


class OmdbMovie(BaseModel):
    """Normalized subset of fields we care about from the OMDb API response."""

    title: str
    year: Optional[str] = None
    genre: Optional[str] = None
    plot: Optional[str] = None
    poster: Optional[str] = None
    imdb_rating: Optional[str] = None
    imdb_id: Optional[str] = None
    director: Optional[str] = None
    actors: Optional[str] = None
    runtime: Optional[str] = None
    found: bool = True
    error: Optional[str] = None

    @classmethod
    def from_omdb_json(cls, data: dict) -> "OmdbMovie":
        if data.get("Response") == "False":
            return cls(
                title=data.get("_query_title", "Unknown"),
                found=False,
                error=data.get("Error", "Not found"),
            )
        return cls(
            title=data.get("Title", "Unknown"),
            year=data.get("Year"),
            genre=data.get("Genre"),
            plot=data.get("Plot"),
            poster=data.get("Poster") if data.get("Poster") != "N/A" else None,
            imdb_rating=data.get("imdbRating"),
            imdb_id=data.get("imdbID"),
            director=data.get("Director"),
            actors=data.get("Actors"),
            runtime=data.get("Runtime"),
            found=True,
        )

In [ ]:
%%writefile omdb_client.py
"""
Thin wrapper around the OMDb API (http://www.omdbapi.com/).

Get a free API key at http://www.omdbapi.com/apikey.aspx
"""

from __future__ import annotations

import requests
from models import OmdbMovie
import tracing

OMDB_BASE_URL = "https://www.omdbapi.com/"


class OmdbClient:
    def __init__(self, api_key: str, timeout: int = 8):
        if not api_key:
            raise ValueError("OMDb API key is required.")
        self.api_key = api_key
        self.timeout = timeout

    def get_movie(self, title: str, year: int | None = None) -> OmdbMovie:
        """Look up a single movie by title (and optional year) for rich details."""
        tracing.log(f"omdb_client.get_movie: looking up title={title!r}, year={year}")
        params = {"apikey": self.api_key, "t": title, "plot": "short"}
        if year:
            params["y"] = str(year)

        try:
            with tracing.timed(f"omdb_client.get_movie: OMDb request for {title!r}"):
                resp = requests.get(OMDB_BASE_URL, params=params, timeout=self.timeout)
                resp.raise_for_status()
                data = resp.json()
        except requests.RequestException as e:
            tracing.log(f"omdb_client.get_movie: request failed for {title!r} - {e}", level="ERROR")
            return OmdbMovie(title=title, found=False, error=str(e))

        data["_query_title"] = title
        movie = OmdbMovie.from_omdb_json(data)
        tracing.log(
            f"omdb_client.get_movie: OMDb found={movie.found}, poster="
            f"{'yes' if movie.poster else 'no'}, error={movie.error!r}"
        )

        # OMDb sometimes returns a poster URL that is dead/unreachable (common for
        # very recently released movies before a real poster is indexed). Verify it
        # actually resolves before we hand it to the UI, so we can fall back cleanly
        # instead of showing a broken-image icon.
        if movie.poster and not self._poster_is_reachable(movie.poster):
            tracing.log(f"omdb_client.get_movie: poster URL unreachable for {title!r}, dropping it")
            movie.poster = None
            movie.error = movie.error or "Poster URL on file is broken/unreachable."

        return movie

    def _poster_is_reachable(self, url: str) -> bool:
        if not url.startswith(("http://", "https://")):
            return False
        try:
            resp = requests.head(url, timeout=4, allow_redirects=True)
            if resp.status_code >= 400:
                return False
            content_type = resp.headers.get("Content-Type", "")
            return content_type.startswith("image/") or content_type == ""
        except requests.RequestException:
            return False

    def search(self, query: str) -> list[dict]:
        """Simple title search - useful for autocomplete-style lookups."""
        params = {"apikey": self.api_key, "s": query}
        try:
            resp = requests.get(OMDB_BASE_URL, params=params, timeout=self.timeout)
            resp.raise_for_status()
            data = resp.json()
        except requests.RequestException:
            return []

        if data.get("Response") == "False":
            return []
        return data.get("Search", [])

In [ ]:
%%writefile recommender.py
"""
LangChain + Gemini logic: turns a chat history into a structured
RecommendationResponse (see models.py) using Gemini's structured output mode.

Gemini's structured-output mode (response_schema / with_structured_output) and its
native Google Search grounding tool cannot be used in the same API call. So for any
request that looks like it needs current/recent info (new releases, "what's out in
2026", etc.) we run two calls:
  1. A grounded, plain-text research call with Google Search enabled, to pull real
     current titles.
  2. A structured call that formats those (or the model's own knowledge, for
     evergreen requests) into a validated RecommendationResponse.
"""

from __future__ import annotations

import datetime as _dt
from typing import List, Optional, Tuple

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

from models import RecommendationResponse
import tracing

# The new consolidated google-genai SDK (used by langchain-google-genai>=4.0) wants
# its own Tool/GoogleSearch objects rather than a bare {"google_search": {}} dict -
# the old dict form is a holdover from the legacy SDK and may get silently ignored.
try:
    from google.genai.types import GoogleSearch as _GoogleSearch
    from google.genai.types import Tool as _GenAITool

    _SEARCH_TOOL = _GenAITool(google_search=_GoogleSearch())
except ImportError:  # pragma: no cover - fallback for older SDKs
    _SEARCH_TOOL = {"google_search": {}}

TODAY_STR = _dt.date.today().strftime("%B %d, %Y")

BASE_SYSTEM_PROMPT = f"""\
You are Gemini Cinema Advisor, an enthusiastic and knowledgeable film critic \
helping a user find movies to watch. Today's date is {TODAY_STR}.

Guidelines:
- Ask brief clarifying questions if you don't have enough to go on yet (favorite \
genres, mood, eras, actors/directors they love, or things they hate).
- Once you have enough signal, recommend up to 5 real, existing movies that fit.
- Never invent a movie title. Only recommend real films you are confident exist.
- Give each recommendation a short, specific reason tied to what the user said - \
avoid generic blurbs.
- Keep the conversational "reply" field short (1-3 sentences).
- If the user says they hate a theme/genre, never recommend movies in it again.
"""

SYSTEM_TEMPLATE = BASE_SYSTEM_PROMPT + "\n\n{research_block}"

RESEARCH_SYSTEM_PROMPT = f"""\
You are a film research assistant. Today's date is {TODAY_STR}. Use Google Search \
to find real, currently-released or recently-announced movies that match the \
user's request below - especially if they're asking about new/recent/2025/2026 \
releases, which may be after your own training data.

Return a short plain-text list: title, year, and a one-sentence note per movie \
(cast, plot, genre - whatever is relevant). Only include movies you are confident \
are real, based on the search results. If the request doesn't need current info \
(e.g. it's about a classic era or genre in general), just say "No research needed."
"""


def _needs_research(user_input: str) -> bool:
    """Heuristic: does this request likely need up-to-date/current info?"""
    keywords = [
        "2024", "2025", "2026", "2027", "new", "recent", "latest", "just released",
        "this year", "currently", "now playing", "in theaters", "streaming now",
        "upcoming",
    ]
    lowered = user_input.lower()
    return any(k in lowered for k in keywords)


def _is_gemini_3_plus(model_name: str) -> bool:
    """Gemini 3.x+ models no longer accept/recommend temperature/top_p/top_k -
    they're tuned for their default reasoning settings, so we skip passing it."""
    return model_name.startswith("gemini-3")


def _make_llm(api_key: str, model_name: str, temperature: float | None = None):
    kwargs = {"model": model_name, "google_api_key": api_key}
    if temperature is not None and not _is_gemini_3_plus(model_name):
        kwargs["temperature"] = temperature
    return ChatGoogleGenerativeAI(**kwargs)


def _research_current_movies(api_key: str, user_input: str, model_name: str) -> str:
    """Grounded (Google Search) plain-text call to surface real current titles."""
    tracing.log(f"recommender._research_current_movies: building search-grounded LLM (model={model_name})")
    research_llm = _make_llm(api_key, model_name, temperature=0.3).bind_tools(
        [_SEARCH_TOOL]
    )

    prompt = ChatPromptTemplate.from_messages(
        [("system", RESEARCH_SYSTEM_PROMPT), ("human", "{user_input}")]
    )
    try:
        with tracing.timed("recommender._research_current_movies: Gemini call w/ google_search tool"):
            result = (prompt | research_llm).invoke({"user_input": user_input})
        content = result.content
        if isinstance(content, str):
            tracing.log(f"recommender._research_current_movies: got string content ({len(content)} chars)")
            return content
        if isinstance(content, list):
            # Some Gemini 3.x responses return content as a list of blocks
            # instead of a plain string - pull out any text parts.
            tracing.log(f"recommender._research_current_movies: got list content ({len(content)} blocks)")
            parts = [
                block.get("text", "")
                for block in content
                if isinstance(block, dict) and block.get("text")
            ]
            return "\n".join(parts) if parts else str(content)
        return str(content)
    except Exception as e:
        # Grounding can fail (older model, unsupported region, etc.) - degrade
        # gracefully rather than breaking the whole recommendation flow, but keep
        # the real error visible for debugging.
        tracing.log_exception("recommender._research_current_movies")
        return f"No research available (search grounding failed: {e})"


def build_chain(api_key: str, model_name: str, temperature: float):
    """Create a LangChain runnable that outputs a validated RecommendationResponse."""
    llm = _make_llm(api_key, model_name, temperature=temperature)
    structured_llm = llm.with_structured_output(RecommendationResponse)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_TEMPLATE),
            MessagesPlaceholder("history"),
            ("human", "{user_input}"),
        ]
    )

    return prompt | structured_llm


def history_to_lc_messages(history: List[Tuple[str, str]]):
    """Convert [(role, text), ...] pairs from Streamlit session state into LC messages."""
    lc_messages = []
    for role, text in history:
        if role == "user":
            lc_messages.append(HumanMessage(content=text))
        else:
            lc_messages.append(AIMessage(content=text))
    return lc_messages


def get_recommendation(
    api_key: str,
    history: List[Tuple[str, str]],
    user_input: str,
    model_name: str = "gemini-3.6-flash",
) -> Tuple[RecommendationResponse, Optional[str]]:
    """Run one turn of the conversation and return (structured_response, research_notes).

    research_notes is None when the research step wasn't triggered for this turn,
    so the caller can distinguish "didn't need to search" from "search ran".
    """
    tracing.log(f"recommender.get_recommendation: called with user_input={user_input!r}, model={model_name}")
    research_notes: Optional[str] = None
    research_block = ""

    needs_research = _needs_research(user_input)
    tracing.log(f"recommender.get_recommendation: _needs_research -> {needs_research}")

    if needs_research:
        research_notes = _research_current_movies(api_key, user_input, model_name)
        research_block = (
            "Up-to-date research notes from Google Search (use these for any "
            "current/recent movie titles - they are more reliable than your training "
            f"data for anything near {TODAY_STR}. If these notes explicitly say "
            "research failed or found nothing, tell the user honestly that you "
            "couldn't confirm current releases, rather than guessing older movies "
            f"and implying they're new):\n{research_notes}"
        )

    chain = build_chain(api_key, model_name, temperature=0.7)
    lc_history = history_to_lc_messages(history)
    # research_block/user_input are passed as template *variables* (not baked into
    # the template string itself), so any literal { or } inside them - e.g. from
    # search result text - is inserted verbatim instead of being re-parsed as a
    # template placeholder, which is what caused the earlier f-string crash.
    try:
        with tracing.timed("recommender.get_recommendation: structured Gemini call"):
            result = chain.invoke(
                {"history": lc_history, "user_input": user_input, "research_block": research_block}
            )
    except Exception:
        tracing.log_exception("recommender.get_recommendation (structured call)")
        raise

    tracing.log(
        f"recommender.get_recommendation: got {len(result.movies)} movie(s) back, "
        f"follow_up={'yes' if result.follow_up_question else 'no'}"
    )
    return result, research_notes

In [ ]:
%%writefile evaluator.py
"""
An independent "judge" call that scores how well each recommended movie (and the
set as a whole) actually fits what the user asked for.

This is intentionally a *separate* Gemini call rather than asking the recommender
to grade its own homework in the same response - a fresh, critically-instructed
call gives a more honest signal than self-reported confidence.
"""

from __future__ import annotations

from typing import List, Optional, Tuple

from langchain_core.prompts import ChatPromptTemplate

from models import EvaluationResponse, MovieRecommendation
from recommender import _make_llm  # reuse the gemini-3-aware LLM builder
import tracing

EVALUATOR_SYSTEM_PROMPT = """\
You are a strict, independent film-recommendation critic. You did NOT make these \
recommendations - another assistant did, and your job is to judge them honestly.

For the user's request below and the list of recommended movies, score:
- Each movie individually (1-10): does it genuinely fit what the user asked for? \
A generic "good movie" that ignores their specific ask should score low, even if \
it's a great film in general.
- The set overall (1-10): does the combination cover the request well, or is it \
repetitive/off-target/too safe?

Be genuinely critical - do not default to high scores. A perfect fit is rare.
Keep justifications to one short, specific sentence each.
"""

# The user's request and the movie list are passed in as template *variables*,
# not concatenated into the prompt string, so any literal { or } in a movie title
# or reason doesn't get mis-parsed as a template placeholder.
EVALUATOR_TEMPLATE = EVALUATOR_SYSTEM_PROMPT + "\n\nUser's request: {user_request}\n\nRecommended movies:\n{movie_list}"


def _movies_to_plain_text(movies: List[MovieRecommendation]) -> str:
    lines = []
    for m in movies:
        year_str = f" ({m.year})" if m.year else ""
        genres_str = f" [{', '.join(m.genres)}]" if m.genres else ""
        lines.append(f"- {m.title}{year_str}{genres_str}: {m.reason}")
    return "\n".join(lines) if lines else "(no movies were recommended)"


def build_evaluator_chain(api_key: str, model_name: str):
    llm = _make_llm(api_key, model_name, temperature=0.2)
    structured_llm = llm.with_structured_output(EvaluationResponse)
    prompt = ChatPromptTemplate.from_messages([("system", EVALUATOR_TEMPLATE)])
    return prompt | structured_llm


def get_evaluation(
    api_key: str,
    user_request: str,
    movies: List[MovieRecommendation],
    model_name: str = "gemini-3.6-flash",
) -> Tuple[Optional[EvaluationResponse], Optional[str]]:
    """Score the recommendation set. Returns (evaluation, error) - evaluation is
    None on failure (so a broken evaluator never takes down the main recommendation
    flow), but the real error is returned too instead of being swallowed, so the
    caller can surface it for debugging."""
    tracing.log(f"evaluator.get_evaluation: called with {len(movies)} movie(s), model={model_name}")
    if not movies:
        tracing.log("evaluator.get_evaluation: no movies to evaluate, skipping")
        return None, None
    try:
        chain = build_evaluator_chain(api_key, model_name)
        movie_list_text = _movies_to_plain_text(movies)
        with tracing.timed("evaluator.get_evaluation: structured judge call"):
            result = chain.invoke({"user_request": user_request, "movie_list": movie_list_text})
        tracing.log(
            f"evaluator.get_evaluation: overall_score={result.overall_score}, "
            f"{len(result.movie_evaluations)} movie score(s) returned"
        )
        return result, None
    except Exception as e:
        tracing.log_exception("evaluator.get_evaluation")
        return None, str(e)

## Keys

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

GEMINI = get_secret("GEMINI_API_KEY", hint="(aistudio.google.com/apikey) ")
OMDB = get_secret("OMDB_API_KEY", hint="(omdbapi.com/apikey.aspx) ")

## Ask for recommendations
History is a list of `(role, text)` tuples. Words like *new*, *latest*, *2026* trigger the extra Google-Search research call.

In [ ]:
import sys; sys.path.insert(0, "/content/movie_advisor")
from recommender import get_recommendation
from omdb_client import OmdbClient
from evaluator import get_evaluation

MODEL_NAME = "gemini-3.5-flash-lite"      # or "gemini-2.5-flash"
history = []
user_input = "I love slow-burn sci-fi like Arrival and hate jump-scare horror. Suggest 3 films."
result, research = get_recommendation(GEMINI, history, user_input, MODEL_NAME)
print(result.reply, "\n")
omdb = OmdbClient(OMDB)
for m in result.movies:
    info = omdb.get_movie(m.title, m.year)
    print(f"* {m.title} ({m.year}) - IMDb {info.imdb_rating} | {info.runtime} | {info.genre}\n    why: {m.reason}")
if result.follow_up_question:
    print("\nFollow-up:", result.follow_up_question)

## Judge the answer (separate call)

In [ ]:
evaluation, err = get_evaluation(GEMINI, user_input, result.movies, MODEL_NAME)
print(err or "")
if evaluation:
    print("Overall:", evaluation.overall_score, "/10 -", evaluation.overall_feedback)
    for e in evaluation.movie_evaluations:
        print(" -", e.title, e.relevance_score, "/10:", e.justification)

## Why this design?
* `with_structured_output(Model)` = no regex parsing; Pydantic **validates**.
* Structured output and Google Search **can't share one call**, so research and formatting are two calls.
* The judge is a *fresh* call so it can't grade its own homework.
* `tracing.py` logs each step and its timing — handy for debugging.